# 09 · Robustness to missing temperature (E11)

**Goal:** measure how much missing temperature affects SOH estimation, and whether the way it is handled matters (decision D13).

1. **Natural missingness:** test errors on cycles whose temperature is missing vs. complete cycles.
2. **Handling strategies**, each with the same validation-only selection:

| Variant | Handling |
|---|---|
| `impute_indicator` | Training median + missing indicator (default everywhere else) |
| `impute_only` | Training median; the model cannot see which values were imputed |
| `carry_forward` | The battery's last *earlier* temperature (causal), default handling for leading gaps |
| `drop_temperature` | Temperature features removed (is temperature useful at all?) |

3. **Stress test:** additional temperature values are **removed** from the test periods (never added), either scattered (`random`) or as one contiguous outage per battery (`block`), at increasing rates, and error is recomputed. Models are not refitted.

Models: Ridge, MLP and TFT on the E1 within-group split (notebook 05). For the TFT, masked inputs are applied to the cycle history its windows are read from, so perturbations reach every step of its window.

Set `EXPERIMENT_PROFILE=smoke` for a fast code check; smoke results are not for reporting.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repository root whether the kernel starts in the repo root or in notebooks/.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data_pipeline as dp

DATA_ROOT = dp.resolve_data_root(os.environ.get("DATA_ROOT", "data"), REPO_ROOT)
RESULTS_DIR = REPO_ROOT / "results"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("DATA_ROOT env :", os.environ.get("DATA_ROOT", "<unset, default 'data'>"))
print("DATA_ROOT dir exists:", DATA_ROOT.is_dir())

import matplotlib.pyplot as plt

# Fixed colour per battery group (never re-assigned by rank or filter).
GROUP_COLORS = dict(zip(dp.BATTERY_GROUPS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3dd",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#8a8981",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "lines.linewidth": 2,
})
FIG_DIR = RESULTS_DIR / "figures"


def savefig(fig, name):
    """Save a figure for the paper/report (results/figures is regenerated, not committed)."""
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

from src import evaluation as ev
from src import experiments as X
from src import robustness as RB

PROFILE = os.environ.get("EXPERIMENT_PROFILE", "paper")
CONFIG = {
    "profile": PROFILE,
    "models": ["ridge", "mlp", "tft"],
    "variants": list(RB.HANDLING_VARIANTS),
    "seed": 0,
    "train_frac": 0.70,
    "val_frac": 0.15,
    "neural_kwargs": {"epochs": 200, "patience": 20} if PROFILE == "paper" else {"epochs": 8, "patience": 3},
    "mask_rates": [0.0, 0.1, 0.3, 0.5, 1.0] if PROFILE == "paper" else [0.0, 0.5, 1.0],
    "mask_modes": ["random", "block"],
    "mask_seeds": [0, 1, 2] if PROFILE == "paper" else [0],
}
VARIANT_COLORS = dict(zip(RB.HANDLING_VARIANTS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))
MODEL_TITLES = {"ridge": "Ridge", "mlp": "MLP", "tft": "TFT"}
print(CONFIG)

## 2. Load data

In [ ]:
dp.check_data_files(DATA_ROOT, required=["ml_training_dataset.csv"])
data = dp.read_dataset(DATA_ROOT / "ml_training_dataset.csv")
data[dp.GROUP_COL] = data[dp.GROUP_COL].astype(str)
groups = [g for g in dp.BATTERY_GROUPS if g in set(data[dp.GROUP_COL])]
features = dp.model_feature_columns(data)
print("rows:", len(data), "| groups:", groups)

## 3. Natural missingness in training and test periods

Share of cycles with any temperature missing, per group and period. If a group's test period has no missing temperature, its natural-missingness comparison is empty and only the stress test informs it.

In [ ]:
_, _, tests0 = X.pooled_split(data, groups, CONFIG["train_frac"], CONFIG["val_frac"])
test_rows = RB.rows_in(data, pd.concat(tests0.values(), ignore_index=True))
missing = RB.temperature_missing(data)
natural = pd.DataFrame({
    "train+val cycles": (~test_rows).groupby(data[dp.GROUP_COL]).sum(),
    "train+val missing %": (missing & ~test_rows).groupby(data[dp.GROUP_COL]).sum() / (~test_rows).groupby(data[dp.GROUP_COL]).sum() * 100,
    "test cycles": test_rows.groupby(data[dp.GROUP_COL]).sum(),
    "test missing %": (missing & test_rows).groupby(data[dp.GROUP_COL]).sum() / test_rows.groupby(data[dp.GROUP_COL]).sum() * 100,
}).reindex(groups)
natural.round(2)

## 4. Fit every model under every handling variant

Each (variant, model) pair is fitted on the pooled training periods with its own validation-only selection (Ridge alpha; MLP/TFT learning rate from the same grid).

In [ ]:
fitted, fit_info, variant_data = {}, {}, {}
for variant in CONFIG["variants"]:
    vdata, vfeatures, indicators = RB.prepare_variant(variant, data, features)
    variant_data[variant] = (vfeatures, indicators)
    train, val, _ = X.pooled_split(vdata, groups, CONFIG["train_frac"], CONFIG["val_frac"])
    for model in CONFIG["models"]:
        predict, cfg = X.fit_model(model, train, val, vfeatures, CONFIG["seed"], CONFIG["neural_kwargs"],
                                   history=vdata, indicators=indicators)
        fitted[(variant, model)] = predict
        fit_info[f"{variant}/{model}"] = {k: cfg.get(k) for k in ("alpha", "lr", "lr_at_grid_edge")}
display(pd.DataFrame(fit_info).T)

## 5. Natural missingness: errors on cycles with vs. without temperature

Test RMSE pooled over groups, split by whether the cycle's temperature was missing in the original data.

In [ ]:
def variant_inputs(variant, frame):
    """Apply a variant's causal/row-wise transform to a full-data frame; return (test rows, history)."""
    vdata, _, _ = RB.prepare_variant(variant, frame, features)
    return vdata[RB.rows_in(vdata, pd.concat(tests0.values(), ignore_index=True))], vdata

strat_rows, per_group_rows = [], []
for (variant, model), predict in fitted.items():
    vtest, vhistory = variant_inputs(variant, data)
    pred = predict(vtest, history=vhistory)
    flag = missing.loc[vtest.index]
    s = RB.stratified_errors(vtest, pred, flag).assign(variant=variant, model=model)
    per_group_rows.append(s)
    overall = dp.regression_metrics(vtest["SOH_Percent"], pred)
    per_group_rows.append(pd.DataFrame([{dp.GROUP_COL: "ALL", "subset": "all cycles", "n": overall["n"],
                                         "RMSE": overall["RMSE"], "MAE": overall["MAE"], "variant": variant, "model": model}]))
natural_errors = pd.concat(per_group_rows, ignore_index=True)
pooled = natural_errors[natural_errors[dp.GROUP_COL] == "ALL"].pivot_table(
    index=["model", "variant"], columns="subset", values=["RMSE", "n"]).reindex(
    pd.MultiIndex.from_product([CONFIG["models"], CONFIG["variants"]], names=["model", "variant"]))
pooled.round(3)

## 6. Does the handling variant matter?

Per-group test RMSE of each variant minus the default (`impute_indicator`), paired over the six groups (positive = worse than the default). Holm correction within each model. With six groups the smallest attainable two-sided p-value is 0.031, so effect sizes and per-group values carry as much weight as p-values.

In [ ]:
# Per-group RMSE over all test cycles of the group:
rows = []
for (variant, model), predict in fitted.items():
    vtest, vhistory = variant_inputs(variant, data)
    pred = pd.Series(predict(vtest, history=vhistory), index=vtest.index)
    for g, sub in vtest.groupby(dp.GROUP_COL):
        rows.append({"variant": variant, "model": model, dp.GROUP_COL: g,
                     "RMSE": dp.regression_metrics(sub["SOH_Percent"], pred.loc[sub.index])["RMSE"]})
by_group = pd.DataFrame(rows).pivot_table(index=dp.GROUP_COL, columns=["model", "variant"], values="RMSE").reindex(groups)
comparisons = []
for model in CONFIG["models"]:
    base = by_group[(model, "impute_indicator")].to_numpy()
    block, pvals = [], {}
    for variant in CONFIG["variants"][1:]:
        res = ev.paired_comparison(by_group[(model, variant)].to_numpy(), base)
        block.append({"model": model, "variant": variant, **res})
        pvals[variant] = res["p_value"]
    adj = ev.holm_correction(pvals)
    comparisons += [{**r, "p_holm": adj[r["variant"]]} for r in block]
handling_comparison = pd.DataFrame(comparisons)
display(by_group.round(3))
handling_comparison[["model", "variant", "n", "mean_diff", "ci_low", "ci_high", "p_value", "p_holm", "rank_biserial"]].round(4)

## 7. Stress test: removing temperature from the test periods

For each rate, mode and seed, temperature is removed from that share of each battery's test cycles (on the original data), the variant's transform is re-applied (so carry-forward can only use earlier, possibly also removed, values), and the fitted models predict again. `drop_temperature` does not use temperature and is flat by construction; it is the reference line.

In [ ]:
stress_rows = []
for mode in CONFIG["mask_modes"]:
    for rate in CONFIG["mask_rates"]:
        for seed in CONFIG["mask_seeds"]:
            masked_full, masked_rows = RB.mask_temperature(data, test_rows, rate, mode, seed)
            for (variant, model), predict in fitted.items():
                vtest, vhistory = variant_inputs(variant, masked_full)
                pred = pd.Series(predict(vtest, history=vhistory), index=vtest.index)
                m_all = dp.regression_metrics(vtest["SOH_Percent"], pred)
                hit = masked_rows.loc[vtest.index]
                m_hit = dp.regression_metrics(vtest.loc[hit, "SOH_Percent"], pred[hit]) if hit.any() else {"RMSE": np.nan}
                stress_rows.append({"mode": mode, "rate": rate, "seed": seed, "variant": variant, "model": model,
                                    "RMSE_all_test": m_all["RMSE"], "RMSE_masked_cycles": m_hit["RMSE"],
                                    "effective_missing_%": float(RB.temperature_missing(vtest).mean() * 100)})
stress = pd.DataFrame(stress_rows)
stress_summary = stress.groupby(["mode", "model", "variant", "rate"])[["RMSE_all_test", "effective_missing_%"]].mean()
stress_summary.round(3)

In [ ]:
fig, axes = plt.subplots(len(CONFIG["mask_modes"]), len(CONFIG["models"]),
                         figsize=(4.2 * len(CONFIG["models"]), 3.3 * len(CONFIG["mask_modes"])), squeeze=False, sharex=True)
for r, mode in enumerate(CONFIG["mask_modes"]):
    for c, model in enumerate(CONFIG["models"]):
        ax = axes[r][c]
        for variant in CONFIG["variants"]:
            sub = stress[(stress["mode"] == mode) & (stress["model"] == model) & (stress["variant"] == variant)]
            agg = sub.groupby("rate")["RMSE_all_test"].agg(["mean", "min", "max"])
            ax.plot(agg.index * 100, agg["mean"], marker="o", markersize=5, color=VARIANT_COLORS[variant], label=variant)
            ax.fill_between(agg.index * 100, agg["min"], agg["max"], color=VARIANT_COLORS[variant], alpha=0.15, linewidth=0)
        ax.set_title(f"{MODEL_TITLES[model]} · {mode} removal", loc="left", fontsize=10)
        if c == 0:
            ax.set_ylabel("Test RMSE (SOH pp)")
        if r == len(CONFIG["mask_modes"]) - 1:
            ax.set_xlabel("Test cycles with temperature removed (%)")
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False)
fig.suptitle("Stress test: error as temperature is removed from test periods (band = range over seeds)")
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

## 8. Save results

In [ ]:
out = X.write_results(RESULTS_DIR / "missing_temperature.json", {
    "profile": PROFILE, "config": CONFIG, "fit_info": fit_info, "natural_missingness": natural.reset_index(),
    "natural_errors": natural_errors, "by_group_rmse": by_group.stack(["model", "variant"], future_stack=True).rename("RMSE").reset_index(),
    "handling_comparison": handling_comparison, "stress": stress,
}, REPO_ROOT)
print("wrote", out.relative_to(REPO_ROOT))
if PROFILE != "paper":
    print("NOTE: smoke profile — these numbers are for code checks only.")

## 9. Interpretation and limitations

* **Natural missingness** compares different cycles (with vs. without temperature), which may also differ in other ways (e.g. a sensor outage during a specific test phase). The stress test removes values from the *same* cycles, so it isolates the effect of the missing input.
* **Removal patterns** are random or one contiguous block; the real mechanism may differ (notebook 02 reports run lengths). Block removal is closer to the observed pattern if runs are long.
* **`drop_temperature`** being as good as the default would mean temperature adds little here, which is itself a finding worth reporting.
* **`impute_only` vs. `impute_indicator`** isolates the value of telling the model which inputs were imputed; for the TFT this corresponds to its explicit missingness channel.
* Models are not refitted under removal, which reflects deployment: a model trained on the available data meets inputs with more missing values.